# CardioIA – Ir Além 2: Diagnóstico visual de ECG com Rede Neural (MLP)

**Objetivo:** treinar uma rede neural do tipo **MLP (Perceptron Multicamadas)** com **Keras** para classificar batimentos de ECG como **normal** ou **anormal** (classificação binária), a partir de **imagens em tons de cinza**.

**Dataset recomendado no enunciado:** [Heartbeat (Kaggle, Shayan Fazeli)](https://www.kaggle.com/datasets/shayanfazeli/heartbeat), com batimentos do MIT-BIH. Cada batimento vem como uma linha de 187 valores (o sinal do ECG) e um rótulo:

| Rótulo original | Significado | Nesta atividade |
|---|---|---|
| 0 | Batimento normal (N) | **normal** |
| 1 | Supraventricular ectópico (S) | **anormal** |
| 2 | Ventricular ectópico (V) | **anormal** |
| 3 | Fusão (F) | **anormal** |
| 4 | Não classificável (Q) | **anormal** |

**Etapas:** (1) carregar dados → (2) transformar cada batimento em uma **imagem** em tons de cinza → (3) pré-processar → (4) criar o MLP → (5) treinar → (6) testar e avaliar.

> ⚠️ Projeto acadêmico. Não substitui avaliação médica.

## 1. Importações e configurações

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings("ignore")

SEMENTE = 42                # reprodutibilidade
TAMANHO = 64                # as imagens terão 64 x 64 pixels
N_POR_CLASSE_TREINO = 6000  # amostra balanceada para o treino (o dataset original é muito desbalanceado)
np.random.seed(SEMENTE)

## 2. Carregando o dataset

O código abaixo procura os arquivos `mitbih_train.csv` e `mitbih_test.csv` na pasta atual, em `./data` e, se estiver no Colab/Kaggle, tenta baixar via `kagglehub`.

**Se o download automático falhar:** baixe o dataset no Kaggle (botão *Download*), extraia e coloque `mitbih_train.csv` e `mitbih_test.csv` na mesma pasta deste notebook (no Colab, use o ícone de pasta à esquerda e arraste os arquivos).

In [ ]:
def localizar_arquivos():
    for pasta in [".", "data", "/kaggle/input/heartbeat"]:
        treino, teste = os.path.join(pasta, "mitbih_train.csv"), os.path.join(pasta, "mitbih_test.csv")
        if os.path.exists(treino) and os.path.exists(teste):
            return treino, teste
    try:
        import kagglehub                       # pip install kagglehub
        pasta = kagglehub.dataset_download("shayanfazeli/heartbeat")
        treino = glob.glob(os.path.join(pasta, "**", "mitbih_train.csv"), recursive=True)[0]
        teste = glob.glob(os.path.join(pasta, "**", "mitbih_test.csv"), recursive=True)[0]
        return treino, teste
    except Exception as erro:
        raise FileNotFoundError(
            "Não encontrei os CSVs. Baixe o dataset no Kaggle e coloque mitbih_train.csv "
            "e mitbih_test.csv ao lado deste notebook."
        ) from erro

arq_treino, arq_teste = localizar_arquivos()
print("Treino:", arq_treino)
print("Teste :", arq_teste)

# header=None: os CSVs não têm cabeçalho. Colunas 0..186 = sinal; coluna 187 = rótulo
df_treino = pd.read_csv(arq_treino, header=None)
df_teste = pd.read_csv(arq_teste, header=None)
print("Formato treino:", df_treino.shape, "| teste:", df_teste.shape)

In [ ]:
# Separar sinais (X) e rótulos (y) e transformar em problema BINÁRIO: 0 = normal, 1 = anormal
sinais_treino = df_treino.iloc[:, :-1].to_numpy(dtype="float32")
sinais_teste = df_teste.iloc[:, :-1].to_numpy(dtype="float32")
y_treino_orig = df_treino.iloc[:, -1].astype(int).to_numpy()
y_teste_orig = df_teste.iloc[:, -1].astype(int).to_numpy()

y_treino_bin = (y_treino_orig != 0).astype("int32")   # 0 = normal, 1 = anormal
y_teste_bin = (y_teste_orig != 0).astype("int32")

print("Treino -> normal:", (y_treino_bin == 0).sum(), "| anormal:", (y_treino_bin == 1).sum())
print("Teste  -> normal:", (y_teste_bin == 0).sum(), "| anormal:", (y_teste_bin == 1).sum())

### Olhando alguns batimentos

Cada linha é um batimento (187 pontos no tempo, valores entre 0 e 1).

In [ ]:
fig, eixos = plt.subplots(2, 4, figsize=(14, 5), sharey=True)
for classe, linha in zip([0, 1], eixos):
    indices = np.where(y_treino_bin == classe)[0][:4]
    for i, ax in zip(indices, linha):
        ax.plot(sinais_treino[i], color="crimson" if classe else "navy")
        ax.set_title("normal" if classe == 0 else "anormal")
        ax.set_xticks([])
plt.suptitle("Exemplos de batimentos (sinal bruto)")
plt.tight_layout()
plt.show()

## 3. Transformando cada sinal em uma imagem em tons de cinza

O MLP do enunciado trabalha com **imagens**. Cada batimento será desenhado como uma imagem **64 × 64** em tons de cinza: fundo preto (0) e traçado do ECG branco (255). A função faz o desenho com NumPy (rápida, sem usar o matplotlib para milhares de gráficos).

Isso já inclui o pré-processamento pedido: **redimensionar** (tamanho fixo) e **tons de cinza** (um único canal).

In [ ]:
def sinal_para_imagem(sinais, tamanho=TAMANHO, lote=2000):
    """Converte sinais (N, 187) com valores em [0,1] em imagens (N, tamanho, tamanho) uint8 (0-255)."""
    saida = []
    for inicio in range(0, len(sinais), lote):
        parte = np.clip(sinais[inicio:inicio + lote], 0, 1)
        n, pontos = parte.shape
        # reamostra o sinal em 4 passos por coluna da imagem
        posicoes = np.linspace(0, pontos - 1, tamanho * 4 + 1)
        baixo = np.floor(posicoes).astype(int)
        alto = np.minimum(baixo + 1, pontos - 1)
        frac = posicoes - baixo
        fino = parte[:, baixo] * (1 - frac) + parte[:, alto] * frac       # (n, tamanho*4)
        # cada coluna usa 5 amostras (compartilhando a borda com a vizinha) para o traçado não ficar quebrado
        fino = np.stack([fino[:, j:j + tamanho * 4:4] for j in range(5)], axis=2)   # (n, tamanho, 5)
        topo = ((1 - fino.max(axis=2)) * (tamanho - 1)).round().astype(int)     # linha do ponto mais alto
        base = ((1 - fino.min(axis=2)) * (tamanho - 1)).round().astype(int)     # linha do ponto mais baixo
        linhas = np.arange(tamanho)[None, :, None]
        mascara = (linhas >= topo[:, None, :]) & (linhas <= base[:, None, :])    # (n, linhas, colunas)
        saida.append((mascara * 255).astype("uint8"))
    return np.concatenate(saida)

# Teste rápido: converter 1 batimento normal e 1 anormal
exemplos = sinal_para_imagem(np.stack([
    sinais_treino[np.where(y_treino_bin == 0)[0][0]],
    sinais_treino[np.where(y_treino_bin == 1)[0][0]],
]))
fig, eixos = plt.subplots(1, 2, figsize=(6, 3))
for ax, img, titulo in zip(eixos, exemplos, ["normal", "anormal"]):
    ax.imshow(img, cmap="gray"); ax.set_title(f"{titulo} ({img.shape[0]}x{img.shape[1]})"); ax.axis("off")
plt.show()

### Montando os conjuntos de treino e teste

- **Treino:** o dataset original é muito desbalanceado (a maioria é normal). Para o modelo não "aprender a chutar sempre normal", usamos uma **amostra balanceada** (mesma quantidade de normais e anormais).
- **Teste:** usamos o arquivo de teste **completo**, sem mexer na proporção, para avaliar em condições mais realistas.

In [ ]:
rng = np.random.default_rng(SEMENTE)
idx_normal = rng.choice(np.where(y_treino_bin == 0)[0], N_POR_CLASSE_TREINO, replace=False)
idx_anormal = rng.choice(np.where(y_treino_bin == 1)[0], min(N_POR_CLASSE_TREINO, (y_treino_bin == 1).sum()), replace=False)
idx = rng.permutation(np.concatenate([idx_normal, idx_anormal]))

X_treino = sinal_para_imagem(sinais_treino[idx])
y_treino = y_treino_bin[idx]
X_teste = sinal_para_imagem(sinais_teste)
y_teste = y_teste_bin

print("X_treino:", X_treino.shape, X_treino.dtype, "| normais:", (y_treino == 0).sum(), "| anormais:", (y_treino == 1).sum())
print("X_teste :", X_teste.shape, "| normais:", (y_teste == 0).sum(), "| anormais:", (y_teste == 1).sum())

### (Opcional) Usar imagens reais de ECG, como as da Fase 1

Se preferir usar **fotos/imagens de ECG** em pastas (por exemplo, o dataset do Mendeley usado na Fase 1), a função abaixo faz o pré-processamento clássico: **abre a imagem, converte para tons de cinza, redimensiona e normaliza**. Organize as pastas assim: `pasta/normal/*.png` e `pasta/anormal/*.png`. Esta célula só roda se você apontar uma pasta.

In [ ]:
from PIL import Image

def carregar_imagens_pasta(pasta, tamanho=TAMANHO):
    """Lê pasta/normal e pasta/anormal; devolve imagens (N, t, t) uint8 em cinza e rótulos 0/1."""
    imagens, rotulos = [], []
    for rotulo, nome in enumerate(["normal", "anormal"]):
        for caminho in glob.glob(os.path.join(pasta, nome, "*")):
            try:
                img = Image.open(caminho).convert("L").resize((tamanho, tamanho))   # cinza + redimensionar
                imagens.append(np.array(img, dtype="uint8"))
                rotulos.append(rotulo)
            except Exception:
                pass   # ignora arquivos que não são imagens
    return np.array(imagens), np.array(rotulos)

PASTA_IMAGENS_PROPRIAS = None     # ex.: "ecg_imagens"  (deixe None para usar apenas o dataset do Kaggle)
if PASTA_IMAGENS_PROPRIAS:
    X_extra, y_extra = carregar_imagens_pasta(PASTA_IMAGENS_PROPRIAS)
    print("Imagens próprias carregadas:", X_extra.shape)
else:
    print("Usando apenas o dataset do Kaggle convertido em imagens.")

## 4. Criando a rede neural MLP com Keras

Um **MLP** é uma rede de camadas totalmente conectadas (`Dense`). Como ele não "enxerga" imagens 2D, a primeira camada `Flatten` transforma a imagem 64×64 em um vetor de 4096 números.

Arquitetura:
1. `Rescaling(1/255)`: normaliza os pixels de 0–255 para 0–1.
2. `Flatten`: imagem → vetor.
3. `Dense(256, relu)` + `Dropout(0.3)`: camada oculta (o Dropout "desliga" neurônios ao acaso no treino para reduzir overfitting).
4. `Dense(128, relu)` + `Dropout(0.3)`.
5. `Dense(1, sigmoid)`: saída entre 0 e 1 = probabilidade do batimento ser **anormal**.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEMENTE)

modelo = keras.Sequential([
    layers.Input(shape=(TAMANHO, TAMANHO)),
    layers.Rescaling(1.0 / 255),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.summary()

## 5. Treinamento

Separamos 20% do treino para **validação** (acompanhar se o modelo está generalizando) e usamos `EarlyStopping` para parar quando a validação deixar de melhorar.

In [ ]:
parada_antecipada = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True
)

historico = modelo.fit(
    X_treino, y_treino,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=[parada_antecipada],
    verbose=2,
)
print("Épocas treinadas:", len(historico.history["loss"]))

In [ ]:
# Curvas de aprendizado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(historico.history["loss"], label="treino"); ax1.plot(historico.history["val_loss"], label="validação")
ax1.set_title("Perda (loss)"); ax1.set_xlabel("época"); ax1.legend()
ax2.plot(historico.history["accuracy"], label="treino"); ax2.plot(historico.history["val_accuracy"], label="validação")
ax2.set_title("Acurácia"); ax2.set_xlabel("época"); ax2.legend()
plt.tight_layout(); plt.show()

## 6. Teste e avaliação

Agora avaliamos no conjunto de **teste** (batimentos que o modelo nunca viu). A saída do modelo é uma probabilidade; usamos o limiar 0,5 para decidir entre normal e anormal.

Em saúde, olhar só a acurácia não basta: o **recall da classe anormal** diz quantos batimentos anormais o modelo realmente encontrou (deixar passar um anormal é o erro mais grave).

In [ ]:
probabilidades = modelo.predict(X_teste, batch_size=256, verbose=0).ravel()
previsoes = (probabilidades >= 0.5).astype("int32")

acuracia = accuracy_score(y_teste, previsoes)
print(f"Acurácia no teste: {acuracia:.2%}\n")
print(classification_report(y_teste, previsoes, target_names=["normal", "anormal"], digits=3))

In [ ]:
matriz = confusion_matrix(y_teste, previsoes)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(matriz, cmap="Reds")
ax.set_xticks([0, 1], ["normal", "anormal"]); ax.set_yticks([0, 1], ["normal", "anormal"])
ax.set_xlabel("Previsto"); ax.set_ylabel("Real"); ax.set_title("Matriz de confusão (teste)")
for (i, j), valor in np.ndenumerate(matriz):
    ax.text(j, i, f"{valor}", ha="center", va="center", color="white" if valor > matriz.max() / 2 else "black")
plt.tight_layout(); plt.show()

In [ ]:
# Exemplos de acertos e erros do modelo
erros = np.where(previsoes != y_teste)[0]
acertos = np.where(previsoes == y_teste)[0]
amostra = list(rng.choice(acertos, 4, replace=False)) + list(rng.choice(erros, min(4, len(erros)), replace=False))

fig, eixos = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(eixos.ravel(), amostra):
    ax.imshow(X_teste[i], cmap="gray")
    real, prev = ("normal", "anormal")[y_teste[i]], ("normal", "anormal")[previsoes[i]]
    ax.set_title(f"real: {real}\nprevisto: {prev} ({probabilidades[i]:.0%})", color="green" if real == prev else "red", fontsize=9)
    ax.axis("off")
plt.suptitle("Linha de cima: acertos | Linha de baixo: erros")
plt.tight_layout(); plt.show()

In [ ]:
# Resumo automático com os números desta execução
vn, fp, fn, vp = matriz.ravel()
recall_anormal = vp / (vp + fn)
precisao_anormal = vp / (vp + fp)
print("=== RESUMO ===")
print(f"Acurácia no teste ........ {acuracia:.2%}")
print(f"Recall do 'anormal' ...... {recall_anormal:.2%}  ({fn} anormais passaram despercebidos de {vp + fn})")
print(f"Precisão do 'anormal' .... {precisao_anormal:.2%}  ({fp} alarmes falsos entre {vp + fp} alertas)")

## 7. Interpretação e limitações

- O MLP recebe a imagem "esticada" em um vetor, então **ignora a vizinhança espacial dos pixels**. Uma **CNN** (rede convolucional) normalmente se sai melhor em imagens e seria o próximo passo natural.
- O treino usou uma amostra **balanceada**, mas o teste mantém a proporção original (muito mais batimentos normais). Por isso a **precisão** da classe anormal e a **acurácia** não contam a mesma história: confira o recall, a precisão e a matriz de confusão, e não apenas a acurácia.
- Os batimentos vêm de um conjunto público pequeno de pacientes (MIT-BIH). O modelo pode não funcionar bem em outros aparelhos, populações ou ruídos de sinal: **viés de representatividade**.
- Transformar sinal em imagem foi uma escolha didática pedida no enunciado; para sinais 1D, redes treinadas diretamente no sinal costumam ser mais simples e eficientes.
- Este modelo é **apenas educacional**. Qualquer uso clínico exigiria validação rigorosa, aprovação regulatória e supervisão médica: a IA **apoia**, nunca substitui, o julgamento do profissional de saúde.